# Boiler-analyse: wat leveren de heat pipes?
Leest de uurstatistieken van collector- en vattemperaturen, de bedrijfsuren van de
solarpomp en de PV-productie. Bepaalt per dag hoe warm het vat wordt, hoeveel warmte de
heat pipes toevoegen, en de verhouding heat-pipe-warmte / PV-opbrengst (heatpipe_factor).
Dagen waarop warmte naar een zwembad is afgevoerd, worden herkend en
uitgesloten van de kalibratie.

In [ ]:
import sqlite3, os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

DB = next(p for p in ["/config/home-assistant_v2.db", "/homeassistant/home-assistant_v2.db"] if os.path.exists(p))
COLLECTOR = "sensor.dl2_001e66000000_temperature_sensor_1"
VAT_ONDER = "sensor.dl2_001e66000000_temperature_sensor_2"
VAT_BOVEN = "sensor.dl2_001e66000000_temperature_sensor_3"
POMP_UREN = "sensor.dl2_001e66000000_operating_hours_relay_1"
PV_TELLER = "sensor.solaredge_lifetime_energy"
KWH_PER_GRAAD = 500 * 4.186 / 3600      # 500 liter water
ZWEMBAD_GRENS = 78                      # boven deze toptemperatuur kan warmte naar het zwembad zijn gegaan
DAGEN = 600
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

def stat(sid, kolom):
    r = con.execute("SELECT id, unit_of_measurement FROM statistics_meta WHERE statistic_id=?", (sid,)).fetchone()
    df = pd.read_sql(f"SELECT start_ts, {kolom} AS v FROM statistics WHERE metadata_id=? AND start_ts>=? ORDER BY start_ts",
                     con, params=(r[0], time.time() - DAGEN * 86400))
    idx = pd.DatetimeIndex(pd.to_datetime(df.start_ts, unit="s", utc=True)).tz_convert("Europe/Amsterdam")
    s = pd.Series(df.v.values, index=idx)
    return s / 1000 if (kolom == "sum" and r[1] == "Wh") else s

u = pd.DataFrame({"coll": stat(COLLECTOR, "mean"), "onder": stat(VAT_ONDER, "mean"),
                  "boven": stat(VAT_BOVEN, "mean"), "boven_max": stat(VAT_BOVEN, "max"),
                  "pomp": stat(POMP_UREN, "mean").diff().clip(0, 1),
                  "pv": stat(PV_TELLER, "sum").diff().clip(0, 10)})
u = u.asfreq("h")
u["gem"] = (u.onder + u.boven) / 2
u["d_energie"] = u.gem.diff() * KWH_PER_GRAAD          # kWh erbij/eraf in het vat per uur
u["zon_aan"] = (u.pomp > 0.1) & (u.coll > u.onder + 3)
print(f"{u.index.min():%d-%m-%Y} t/m {u.index.max():%d-%m-%Y}, {len(u)} uren")

## Per dag
Zonnewarmte = som van de stijgingen in vatenergie in uren waarin de solarpomp draaide en de
collector warmer was dan de onderkant van het vat. Een dag telt als 'zwembad' als het vat
boven de grens kwam of als het vat in zonuren duidelijk afkoelde terwijl de collector leverde.

In [ ]:
d = pd.DataFrame({
    "max_boven": u.boven_max.resample("D").max(),
    "min_ochtend": u.gem.between_time("05:00", "09:00").resample("D").min(),
    "max_gem": u.gem.resample("D").max(),
    "zonwarmte_kwh": u.d_energie.where(u.zon_aan & (u.d_energie > 0), 0).resample("D").sum(),
    "afkoeling_in_zon_kwh": -u.d_energie.where(u.zon_aan & (u.d_energie < -0.3), 0).resample("D").sum(),
    "pomp_uren": u.pomp.resample("D").sum(),
    "pv_kwh": u.pv.resample("D").sum(),
})
d["stijging_graden"] = d.max_gem - d.min_ochtend
d["zwembad"] = (d.max_boven >= ZWEMBAD_GRENS) | (d.afkoeling_in_zon_kwh > 1.5)
d["factor"] = (d.zonwarmte_kwh / d.pv_kwh).where((d.pv_kwh > 3) & (d.pomp_uren > 1))
d = d.dropna(subset=["max_boven"])
d.index = d.index.tz_localize(None)
d.round(2).to_csv("boiler_per_dag.csv", sep=";", decimal=",")
print(len(d), "dagen,", int(d.zwembad.sum()), "dagen mogelijk met warmte naar zwembad")

In [ ]:
maand = d.groupby(d.index.to_period("M")).agg(
    dagen=("max_boven", "size"),
    zwembaddagen=("zwembad", "sum"),
    max_boven_mediaan=("max_boven", "median"),
    max_boven_p90=("max_boven", lambda x: x.quantile(0.9)),
    stijging_mediaan=("stijging_graden", "median"),
    zonwarmte_mediaan=("zonwarmte_kwh", "median"),
    pv_mediaan=("pv_kwh", "median"),
)
schoon = d[~d.zwembad]
maand["factor_mediaan"] = schoon.groupby(schoon.index.to_period("M")).factor.median()
print(maand.round(1).to_string())

In [ ]:
s = schoon.dropna(subset=["factor"])
x = s.max_gem.clip(20, 90)
bins = [20, 40, 50, 60, 70, 90]
per_temp = s.groupby(pd.cut(x, bins)).factor.agg(["count", "median"]).round(2)
print("Factor zonwarmte/PV naar vattemperatuur (dagen zonder zwembad):")
print(per_temp.to_string())
print("\nTotaal mediaan factor:", round(s.factor.median(), 2), "| P25-P75:",
      round(s.factor.quantile(.25), 2), "-", round(s.factor.quantile(.75), 2))

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axs[0].plot(d.index, d.max_boven, color="#c0392b", lw=1, label="max boven")
axs[0].plot(d.index, d.min_ochtend, color="#2a7ab0", lw=1, label="gem. ochtend")
axs[0].scatter(d.index[d.zwembad], d.max_boven[d.zwembad], s=8, color="k", label="zwembad-dag")
axs[0].axhline(ZWEMBAD_GRENS, ls=":", color="gray"); axs[0].set_ylabel("°C"); axs[0].legend()
axs[1].bar(d.index, d.zonwarmte_kwh, color="#e08a2c", label="zonwarmte heat pipes")
axs[1].plot(d.index, d.pv_kwh, color="#2a7ab0", lw=1, label="PV-opbrengst")
axs[1].set_ylabel("kWh/dag"); axs[1].legend(); plt.tight_layout(); plt.show()